In [1]:
import os
from typing import Literal, Optional, List, TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

In [3]:
# ==========================================
# 1. Structured Output Schema (Pydantic)
# ==========================================
class ReviewAnalysis(BaseModel):
    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(
        description="Classify the overall tone of the review based on rating and text."
    )
    draft_response: str = Field(
        description="A professional, brand-appropriate draft response to the review."
    )

In [4]:
# ==========================================
# 2. Graph State Definition
# ==========================================
class ReviewState(TypedDict):
    review_id: str
    author: str
    rating: int
    review_text: str
    sentiment: Optional[str]
    draft_response: Optional[str]
    feedback: Optional[str]
    status: Literal["PENDING", "APPROVED", "REJECTED", "REPHRASING"]

In [5]:
# ==========================================
# 3. Model-Agnostic LLM Initialization
# ==========================================
# Using LangChain's init_chat_model makes this completely model-agnostic.
# You can switch provider to "anthropic" or "openai" just by changing provider and model name.
llm = init_chat_model(
    model="gemini-3.5-flash-lite",
    model_provider="google_genai",
    temperature=0.1
)
structured_llm = llm.with_structured_output(ReviewAnalysis)

In [6]:
# ==========================================
# 4. Graph Nodes
# ==========================================
def classify_and_draft_node(state: ReviewState) -> ReviewState:
    """Analyzes sentiment and drafts a response using Gemini."""
    author = state.get("author", "Customer")
    rating = state.get("rating", 5)
    # Safely check for either key name to prevent KeyError
    review_text = state.get("review_text") or state.get("text", "")
    
    print(f"\n🤖 [Gemini Agent] Processing review from {author}...")
    
    system_prompt = (
        "You are an expert customer success manager for local businesses. "
        "Analyze the incoming customer review by combining the star rating and the text. "
        "Classification rules: "
        "- 1 to 2 stars OR angry/complaining text MUST be classified as 'Negative'. "
        "- 3 stars or mixed/lukewarm text MUST be classified as 'Neutral'. "
        "- 4 to 5 stars with praise MUST be classified as 'Positive'. "
        "Craft a professional, empathetic response draft matching the classified sentiment."    
    )
    
    if state.get("feedback"):
        system_prompt += f"\n\nIMPORTANT REFINEMENT INSTRUCTION FROM ADMIN: {state['feedback']}"

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Author: {author}\nRating: {rating}/5 Stars\nReview: {review_text}")
    ]
    
    result: ReviewAnalysis = structured_llm.invoke(messages)
    
    return {
        **state,
        "sentiment": result.sentiment,
        "draft_response": result.draft_response,
        "status": "PENDING"
    }

In [7]:
# ==========================================
# 5. Build LangGraph Workflow
# ==========================================
workflow = StateGraph(ReviewState)

# Add nodes
workflow.add_node("process_review", classify_and_draft_node)

# Set entry point
workflow.add_edge(START, "process_review")
workflow.add_edge("process_review", END)

# Compile graph with a memory saver for checkpointing / human-in-the-loop
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [10]:
# ==========================================
# Complete Interactive CLI Runner with HITL
# ==========================================

def run_review_agent_cli():
    print("🚀 Initializing Model-Agnostic LangGraph with Gemini & HITL...")
    
    # Initial state for a 2-star review
    state = {
        "review_id": "rev_001",
        "author": "Ananya Deshmukh",
        "rating": 2,
        "review_text": "The espresso was lukewarm and the seating area was overcrowded.",
        "feedback": None,
        "status": "PENDING"
    }
    
    config = {"configurable": {"thread_id": "review_thread_1"}}
    
    while True:
        # Invoke LangGraph node
        output = app.invoke(state, config)
        
        print("\n" + "="*60)
        print(f"📥 Review from {output['author']} ({output['rating']}/5 Stars)")
        print(f"📝 Text: \"{output['review_text']}\"")
        print(f"📊 Sentiment Analysis : [{output['sentiment']}]")
        print(f"✍️  Gemini Draft Response: {output['draft_response']}")
        print("="*60)
        
        # Human-in-the-Loop Action Prompt
        action = input("Actions -> [A]pprove & Send | [R]eject | [F]actor (Rephrase with comments): ").strip().upper()
        
        if action == "A":
            print(f"✅ Success! Response approved and sent to Google Business Profile.")
            break
        elif action == "R":
            print(f"❌ Review dropped. No response sent.")
            break
        elif action == "F":
            user_feedback = input("💬 Enter your refactoring comments/instructions for Gemini: ")
            # Update state with feedback so LangGraph re-runs with instructions
            state = {**output, "feedback": user_feedback, "status": "REPHRASING"}
            print("🔄 Re-executing Gemini agent with your feedback...")
            continue
        else:
            print("⚠️ Invalid choice. Please enter A, R, or F.")

if __name__ == "__main__":
    run_review_agent_cli()

🚀 Initializing Model-Agnostic LangGraph with Gemini & HITL...

🤖 [Gemini Agent] Processing review from Ananya Deshmukh...


/Users/thetsajeet/Documents/workspace/thetsajeet/review-pulse/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📥 Review from Ananya Deshmukh (2/5 Stars)
📝 Text: "The espresso was lukewarm and the seating area was overcrowded."
📊 Sentiment Analysis : [Negative]
✍️  Gemini Draft Response: Dear Ananya, thank you for sharing your feedback with us. We sincerely apologize that your espresso was lukewarm and that the seating area felt overcrowded during your visit. We strive to provide a warm, comfortable experience and quality drinks, and we clearly fell short this time. We are addressing these issues with our team to ensure they do not happen again. We hope you will give us another chance in the future to provide you with the exceptional experience you deserve.
🔄 Re-executing Gemini agent with your feedback...

🤖 [Gemini Agent] Processing review from Ananya Deshmukh...


/Users/thetsajeet/Documents/workspace/thetsajeet/review-pulse/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



📥 Review from Ananya Deshmukh (2/5 Stars)
📝 Text: "The espresso was lukewarm and the seating area was overcrowded."
📊 Sentiment Analysis : [Negative]
✍️  Gemini Draft Response: Hi Ananya, we are truly sorry your espresso was lukewarm and the seating felt overcrowded; we are addressing this immediately.
✅ Success! Response approved and sent to Google Business Profile.
